<div style="display: flex; align-items: center; gap: 8px;">
  <img src="images/microsoft-symbol.svg" alt="Microsoft" style="width: 80px; height: 80px; border-radius: 8px; flex: 0 0 auto;">
  <h1 style="color: #4A2D6F; font-weight: 800; margin: 0;">Microsoft Agent Framework SDK + MCP + Workflows — PoC</h1>
</div>
<p align="center" style="color: #666; font-style: italic; margin-top: -4px;">End-to-end proof of concept — from Agent Framework setup to MCP exposure, workflow orchestration, and Aspire Dashboard tracing</p>

Build and interact with agents using <strong>Microsoft Agent Framework</strong>, <strong>Azure OpenAI</strong>, <strong>Model Context Protocol (MCP)</strong>, and <strong>OpenTelemetry</strong>. This notebook is intentionally <strong>not</strong> using Microsoft Foundry. </br>

The notebook is split into four teaching tracks:

1. <strong>Framework-first setup</strong> — create a local Python environment and install the minimum Agent Framework PoC dependencies.
2. <strong>Observability-first runtime</strong> — turn on Agent Framework instrumentation and send telemetry to an Aspire Dashboard with sensitive data capture enabled for demo purposes.
3. <strong>Agent + tools + MCP</strong> — create a basic teaching agent, then expose a separate Agent Framework agent as an MCP server using the official repo pattern.
4. <strong>Basic workflow</strong> — create a multi-agent workflow so learners can see how orchestration differs from a single-agent run.

For observability, the notebook wires up <strong>OpenTelemetry</strong> so telemetry flows to:

- Aspire Dashboard (primary local viewer)
- Console exporters (secondary local fallback)
- MCP trace propagation metadata when an active span context exists

---

<h2 style="color: #D83B01;">Prerequisites</h2>

<details>
<summary><strong>Expand Prerequisites</strong></summary>

Before running this notebook, ensure the following are in place:

<strong>1. Azure CLI Installed</strong>

The notebook uses <code>AzureCliCredential</code> on purpose because the official <code>microsoft/agent-framework</code> repo recommends explicit Azure CLI auth for samples.

- <strong>Install via winget (Windows):</strong>
  <code>winget install --id Microsoft.AzureCLI -e --accept-source-agreements --accept-package-agreements</code>
- <strong>Other platforms / manual install:</strong> <a href="https://aka.ms/installazurecli">https://aka.ms/installazurecli</a>

<strong>2. Logged in via Azure CLI</strong>

Run:
<code>az login</code><br>
<code>az account show</code>

<strong>3. Azure OpenAI access</strong>

You need an Azure OpenAI endpoint and deployment name. The notebook looks first for:

- <code>AZURE_OPENAI_ENDPOINT</code>
- <code>AZURE_OPENAI_CHAT_MODEL</code>
- <code>AZURE_OPENAI_MODEL</code>

If those are not already set, the notebook will try to reuse <code>azure_openai_endpoint</code> and <code>genai_model</code> from the latest local <code>build_info-&lt;suffix&gt;.json</code> file as a convenience fallback.

<strong>4. Docker Desktop</strong>

The observability path is built around the Aspire Dashboard container. If Docker is available, the notebook can start it for you.

</details>

---

<h2 style="color: #0078D4;">0. Create or Reuse Virtual Environment &amp; Register Kernel</h2>

<ul>
  <li>Create or reuse a project-local <code>.venv</code> for this Agent Framework PoC.</li>
  <li>Install <code>ipykernel</code> in the environment and register it so the notebook can run on the correct interpreter.</li>
</ul>

In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), '.venv')
subprocess.check_call([sys.executable, '-m', 'venv', venv_dir])

venv_python = (
    os.path.join(venv_dir, 'Scripts', 'python.exe')
    if os.name == 'nt'
    else os.path.join(venv_dir, 'bin', 'python')
)

subprocess.check_call([venv_python, '-m', 'pip', 'install', '--upgrade', 'ipykernel'])
subprocess.check_call([
    venv_python,
    '-m',
    'ipykernel',
    'install',
    '--user',
    '--name',
    'agent-framework-sdk-demo',
    '--display-name',
    'Agent Framework SDK Demo (.venv)',
])

print(f'Virtual environment created or reused at {venv_dir}')
print("Select the 'Agent Framework SDK Demo (.venv)' kernel in the top-right kernel picker, then continue.")

<h3 style="color: #D83B01;">Activate or Deactivate the Python Virtual Environment</h3>

Run the next cell to help VS Code discover the <code style="color: #2EA043;">.venv</code> environment if needed.

To deactivate later in a PowerShell terminal, run <code style="color: #D29922;">deactivate</code>.

In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), '.venv')
venv_dir_norm = os.path.normcase(os.path.normpath(venv_dir))
active_venv = os.environ.get('VIRTUAL_ENV', '')
active_venv_norm = os.path.normcase(os.path.normpath(active_venv)) if active_venv else ''

already_active = (
    active_venv_norm == venv_dir_norm
    or os.path.normcase(os.path.normpath(sys.prefix)) == venv_dir_norm
)

if already_active:
    print(f'.venv is already active: {venv_dir}')
else:
    if os.name == 'nt':
        activate_script = os.path.join(venv_dir, 'Scripts', 'Activate.ps1')
        subprocess.check_call(['powershell', '-ExecutionPolicy', 'Bypass', '-File', activate_script])
    else:
        activate_script = os.path.join(venv_dir, 'bin', 'activate')
        subprocess.check_call(['bash', '-c', f'source {activate_script}'])
    print(f'Virtual environment activated: {activate_script}')

print("You can now select the 'Agent Framework SDK Demo (.venv)' kernel in the top-right kernel picker.")

<h2 style="color: #0078D4;">1. Install Python Packages &amp; Dependencies</h2>

Install the packages needed for a framework-first Agent Framework PoC.

<details>
<summary><strong>Package Overview (Expand to view dependencies)</strong></summary>
<br>

🤖 <code style="color: #0078D4; font-weight: 600;">agent-framework-core</code> — the main Agent Framework package; current repo guidance says it includes Azure OpenAI and workflow/orchestration support by default.<br>
🔐 <code style="color: #0078D4; font-weight: 600;">azure-identity</code> — provides <code>AzureCliCredential</code> and async credential support for Azure OpenAI RBAC auth.<br>
🛰️ <code style="color: #0E7C6B; font-weight: 600;">mcp</code> — provides the MCP server plumbing used by the official <code>agent.as_mcp_server()</code> sample pattern.<br>
📡 <code style="color: #0E7C6B; font-weight: 600;">opentelemetry-exporter-otlp-proto-grpc</code> — sends telemetry to the Aspire Dashboard OTLP endpoint.<br>
🧵 <code style="color: #0078D4; font-weight: 600;">anyio</code> — used by the official MCP server sample pattern.<br>
🧪 <code style="color: #0078D4; font-weight: 600;">ipykernel</code> — required so this notebook runs in the project-local virtual environment.

</details>

In [ ]:
import subprocess
import sys

%pip --disable-pip-version-check install --upgrade pip
%pip --disable-pip-version-check install --upgrade agent-framework-core azure-identity mcp anyio ipykernel opentelemetry-exporter-otlp-proto-grpc

print('Agent Framework SDK dependencies installed successfully.')

<h2 style="color: #0078D4;">2. Configure Azure OpenAI and Authentication</h2>

This notebook does <strong>not</strong> use Foundry. It uses Agent Framework with Azure OpenAI directly.

<table style="margin-top: 8px; border: none; border-collapse: collapse;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔐</td>
    <td style="border: none; padding: 4px 0; color: #555;">Uses <code>AzureCliCredential</code> explicitly so auth behavior matches the official repo sample guidance.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">☁️</td>
    <td style="border: none; padding: 4px 0; color: #555;">Reads <code>AZURE_OPENAI_ENDPOINT</code> and <code>AZURE_OPENAI_CHAT_MODEL</code> or falls back to the latest <code>build_info-*.json</code> when available.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📎</td>
    <td style="border: none; padding: 4px 0; color: #555;">Stores the resolved endpoint, model, and async credential in notebook globals for later cells.</td>
  </tr>
</table>

In [4]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from azure.identity.aio import AzureCliCredential

az_exe = 'az.cmd' if os.name == 'nt' else 'az'
if not shutil.which(az_exe):
    raise RuntimeError('Azure CLI is not installed or not on PATH. Install Azure CLI, run az login, and rerun this cell.')

account_check = subprocess.run([az_exe, 'account', 'show'], capture_output=True, text=True, check=False)
if account_check.returncode != 0:
    raise RuntimeError('No active Azure CLI session detected. Run az login in a terminal, then rerun this cell.')

def first_nonempty(*values: str | None) -> str:
    for value in values:
        if value and str(value).strip():
            return str(value).strip()
    return ''

def load_latest_build_info() -> dict:
    candidates = sorted(Path('.').glob('build_info-*.json'), key=lambda item: item.stat().st_mtime, reverse=True)
    if not candidates:
        return {}
    return json.loads(candidates[0].read_text(encoding='utf-8'))

build_info = load_latest_build_info()
azure_openai_endpoint = first_nonempty(
    os.environ.get('AZURE_OPENAI_ENDPOINT'),
    build_info.get('azure_openai_endpoint'),
)
model_name = first_nonempty(
    os.environ.get('AZURE_OPENAI_CHAT_MODEL'),
    os.environ.get('AZURE_OPENAI_MODEL'),
    build_info.get('genai_model'),
)

if not azure_openai_endpoint or not model_name:
    raise RuntimeError(
        'Azure OpenAI configuration is incomplete. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_CHAT_MODEL (or AZURE_OPENAI_MODEL), or keep a build_info-*.json file with azure_openai_endpoint and genai_model values.'
    )

os.environ['AZURE_OPENAI_ENDPOINT'] = azure_openai_endpoint
os.environ['AZURE_OPENAI_CHAT_MODEL'] = model_name
os.environ['AZURE_OPENAI_MODEL'] = model_name

if globals().get('async_credential') is None:
    globals()['async_credential'] = AzureCliCredential()

print('Notebook runtime configured')
print(f'  Azure OpenAI endpoint: {azure_openai_endpoint}')
print(f'  Azure OpenAI deployment: {model_name}')
print('  Authentication: AzureCliCredential (async)')
print('  Runtime mode: Agent Framework + Azure OpenAI (no Foundry)')

Notebook runtime configured
  Azure OpenAI endpoint: https://zolabai-foundry-u49gco.cognitiveservices.azure.com/
  Azure OpenAI deployment: gpt-5.4
  Authentication: AzureCliCredential (async)
  Runtime mode: Agent Framework + Azure OpenAI (no Foundry)


<h2 style="color: #0078D4;">3. Start the Aspire Dashboard</h2>

The official <code>microsoft/agent-framework</code> observability samples recommend OTLP-compatible viewers for local development. For this PoC, Aspire Dashboard is the primary trace viewer.

<details>
<summary><strong>Aspire Dashboard Notes</strong></summary>

- <strong>UI:</strong> <code>http://localhost:18888</code>
- <strong>OTLP gRPC endpoint:</strong> <code>http://localhost:4317</code>
- The notebook will try to start a local Docker container named <code>zolab-agent-framework-aspire</code>.
- Aspire Dashboard frontend login commonly uses a <strong>browser token</strong> for local container runs.
- The next cell now prints the token and a ready-to-open login URL when it can determine them.
- If Docker is not available, the rest of the notebook still works with console exporters only.

</details>

In [5]:
import os
import re
import secrets
import shutil
import subprocess
from urllib.parse import quote

ASPIRE_CONTAINER_NAME = 'zolab-agent-framework-aspire'
ASPIRE_UI_URL = 'http://localhost:18888'
ASPIRE_OTLP_ENDPOINT = 'http://localhost:4317'
ASPIRE_BROWSER_TOKEN = os.environ.get('ASPIRE_DASHBOARD_BROWSER_TOKEN', '').strip()
os.environ['OTEL_EXPORTER_OTLP_ENDPOINT'] = ASPIRE_OTLP_ENDPOINT

if not ASPIRE_BROWSER_TOKEN:
    ASPIRE_BROWSER_TOKEN = secrets.token_hex(16)
    os.environ['ASPIRE_DASHBOARD_BROWSER_TOKEN'] = ASPIRE_BROWSER_TOKEN

LOGIN_URL_PATTERN = re.compile(r'http://localhost:18888/login\?t=([A-Za-z0-9_-]+)')
resolved_browser_token = None
resolved_login_url = None

if not shutil.which('docker'):
    print('Docker was not found on PATH. Aspire Dashboard was not started; console exporters will still work.')
else:
    inspect = subprocess.run(
        ['docker', 'ps', '-a', '--filter', f'name={ASPIRE_CONTAINER_NAME}', '--format', '{{.Names}}\t{{.Status}}'],
        capture_output=True,
        text=True,
        check=False,
    )
    rows = [line.strip() for line in (inspect.stdout or '').splitlines() if line.strip()]

    if rows:
        if 'Up' not in rows[0]:
            subprocess.check_call(['docker', 'start', ASPIRE_CONTAINER_NAME])
            print(f'Started existing Aspire Dashboard container: {ASPIRE_CONTAINER_NAME}')
        else:
            print(f'Aspire Dashboard container is already running: {ASPIRE_CONTAINER_NAME}')
    else:
        subprocess.check_call([
            'docker', 'run', '--rm', '-d',
            '-p', '18888:18888',
            '-p', '4317:18889',
            '-e', 'Dashboard__Frontend__AuthMode=BrowserToken',
            '-e', f'Dashboard__Frontend__BrowserToken={ASPIRE_BROWSER_TOKEN}',
            '--name', ASPIRE_CONTAINER_NAME,
            'mcr.microsoft.com/dotnet/aspire-dashboard:latest',
        ])
        print(f'Created and started Aspire Dashboard container: {ASPIRE_CONTAINER_NAME}')

    logs = subprocess.run(
        ['docker', 'logs', '--tail', '100', ASPIRE_CONTAINER_NAME],
        capture_output=True,
        text=True,
        check=False,
    )
    combined_logs = '\n'.join(part for part in [logs.stdout, logs.stderr] if part)
    match = LOGIN_URL_PATTERN.search(combined_logs)

    if match:
        resolved_browser_token = match.group(1)
        resolved_login_url = f'{ASPIRE_UI_URL}/login?t={quote(resolved_browser_token)}'
    else:
        resolved_browser_token = ASPIRE_BROWSER_TOKEN
        resolved_login_url = f'{ASPIRE_UI_URL}/login?t={quote(resolved_browser_token)}'

print(f'Aspire Dashboard UI: {ASPIRE_UI_URL}')
print(f'OTLP endpoint: {ASPIRE_OTLP_ENDPOINT}')

if resolved_browser_token:
    print('Aspire Dashboard browser token:')
    print(resolved_browser_token)
    print('Open this login URL in your browser:')
    print(resolved_login_url)
    print('If the dashboard prompts for a token, paste the value shown above.')
else:
    print('No browser token was resolved automatically. Run `docker logs --tail 100 zolab-agent-framework-aspire` and use the `/login?t=...` URL printed by the container.')

globals()['ASPIRE_BROWSER_TOKEN'] = resolved_browser_token or ASPIRE_BROWSER_TOKEN
globals()['ASPIRE_LOGIN_URL'] = resolved_login_url

Aspire Dashboard container is already running: zolab-agent-framework-aspire
Aspire Dashboard UI: http://localhost:18888
OTLP endpoint: http://localhost:4317
Aspire Dashboard browser token:
38451cad213438b0919fd19a9baf262c
Open this login URL in your browser:
http://localhost:18888/login?t=38451cad213438b0919fd19a9baf262c
If the dashboard prompts for a token, paste the value shown above.


<h2 style="color: #0078D4;">3.1 Enable Agent Framework Observability</h2>

<details>
<summary><strong>Observability Setup Overview</strong></summary>

Turn on a high-visibility local tracing posture for this PoC:

1. <strong><code>configure_otel_providers(...)</code></strong> — uses the current standard OpenTelemetry environment variables and is the recommended getting-started path in the official repo.
2. <strong><code>ENABLE_INSTRUMENTATION=true</code></strong> — activates Agent Framework telemetry code paths for agents, tools, and workflows.
3. <strong><code>ENABLE_SENSITIVE_DATA=true</code></strong> — includes prompts and responses for this teaching demo. This is intentionally high signal and not a production default.
4. <strong><code>ENABLE_CONSOLE_EXPORTERS=true</code></strong> — keeps a console fallback visible even while Aspire Dashboard is the main viewer.
5. <strong>MCP trace propagation</strong> — the official observability sample notes that Agent Framework automatically propagates active trace context to MCP servers via <code>params._meta</code>.
6. <strong>Manual notebook spans</strong> — the notebook adds a few explicit spans around the key demo runs so learners can see their own orchestration boundaries in the trace tree.

</details>

In [6]:
import logging
import os
from html import escape
from uuid import uuid4

from IPython.display import HTML, display
from agent_framework.observability import configure_otel_providers
from opentelemetry import trace

telemetry_session_id = globals().get('telemetry_session_id') or str(uuid4())
globals()['telemetry_session_id'] = telemetry_session_id

os.environ['ENABLE_INSTRUMENTATION'] = 'true'
os.environ['ENABLE_SENSITIVE_DATA'] = 'true'
os.environ['ENABLE_CONSOLE_EXPORTERS'] = 'true'
os.environ['OTEL_SERVICE_NAME'] = 'zolab-agent-framework-sdk-demo'
os.environ['OTEL_SERVICE_VERSION'] = '2026.04.07'
os.environ['OTEL_RESOURCE_ATTRIBUTES'] = (
    'service.namespace=zolab-agent-framework,'
    f'service.instance.id={telemetry_session_id},'
    'deployment.environment=demo,'
    'demo.type=agent-framework-sdk'
)

logging.basicConfig(
    format='[%(asctime)s - %(pathname)s:%(lineno)d - %(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
logging.getLogger().setLevel(logging.NOTSET)

if not globals().get('_agent_framework_otel_initialized', False):
    configure_otel_providers(enable_console_exporters=True)
    globals()['_agent_framework_otel_initialized'] = True
else:
    print('Agent Framework OpenTelemetry providers were already initialized in this kernel session.')

tracer = trace.get_tracer('zolab.agent_framework_sdk.notebook')
globals()['tracer'] = tracer

accent_style = 'color: #C239B3; font-weight: 700;'
secondary_style = 'color: #0078D4; font-weight: 700;'
display(
    HTML(
        f"""
<div style="font-family: Consolas, 'Cascadia Code', monospace; line-height: 1.5;">
  <div>Observability enabled -&gt; Aspire Dashboard / OTLP for service: '<span style=\"{accent_style}\">{escape(os.environ['OTEL_SERVICE_NAME'])}</span>'</div>
  <div>- OTLP endpoint: <span style=\"{secondary_style}\">{escape(os.environ.get('OTEL_EXPORTER_OTLP_ENDPOINT', 'not set'))}</span></div>
  <div>- Telemetry session id: {escape(telemetry_session_id)}</div>
  <div>- Agent Framework instrumentation: Enabled</div>
  <div>- Sensitive data capture: Enabled for this PoC</div>
  <div>- Console exporters: Enabled</div>
  <div>- MCP trace propagation: Available when MCP tools are invoked under an active span</div>
</div>
"""
    )
)

Agent Framework OpenTelemetry providers were already initialized in this kernel session.


<h2 style="color: #0078D4;">4. Create a Basic Agent with Tools</h2>

This section stays as close as possible to the official Agent Framework repo patterns:

- <strong><code>Agent(...)</code></strong> from the root package
- <strong><code>OpenAIChatClient</code></strong> as the preferred general-purpose client for new work
- <strong><code>@tool</code></strong> for local function tools
- explicit Azure routing via <code>azure_endpoint</code> + <code>AzureCliCredential</code>

In [7]:
from random import randint
from typing import Annotated

from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient

@tool(approval_mode='never_require')
def get_weather(
    location: Annotated[str, Field(description='The location to get the weather for.')],
) -> str:
    conditions = ['sunny', 'cloudy', 'rainy', 'stormy']
    return f'The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C.'

@tool(approval_mode='never_require')
def get_agent_framework_highlights() -> str:
    return (
        'Microsoft Agent Framework focuses on agents, tools, workflows, orchestration patterns, and OpenTelemetry-based observability. '
        'The Python repo shows single-agent patterns, multi-agent orchestrations, MCP examples, and local observability with OTLP exporters.'
    )

@tool(approval_mode='never_require')
def get_observability_checklist() -> str:
    return (
        'Checklist: enable instrumentation, set an OTLP endpoint, choose a viewer like Aspire Dashboard, '
        'turn on sensitive data only in a test environment, and trace both agents and workflows.'
    )

chat_client = globals().get('chat_client')
if chat_client is None:
    chat_client = OpenAIChatClient(
        model=model_name,
        azure_endpoint=azure_openai_endpoint,
        credential=globals()['async_credential'],
    )
    globals()['chat_client'] = chat_client

teaching_agent = Agent(
    client=chat_client,
    name='ZoAgentFrameworkGuide',
    description='A beginner-friendly Agent Framework teaching agent.',
    instructions=(
        'You are a precise but approachable Microsoft Agent Framework teaching assistant. '
        'Use tools when they help. Keep explanations concrete, practical, and focused on agents, tools, MCP, workflows, and observability.'
    ),
    tools=[get_weather, get_agent_framework_highlights, get_observability_checklist],
)
globals()['teaching_agent'] = teaching_agent

print(f'Agent created: {teaching_agent.name}')
print('Attached tools: get_weather, get_agent_framework_highlights, get_observability_checklist')
print(f'Azure OpenAI deployment: {model_name}')

Agent created: ZoAgentFrameworkGuide
Attached tools: get_weather, get_agent_framework_highlights, get_observability_checklist
Azure OpenAI deployment: gpt-5.4


<h3 style="color: #0078D4;">4.1 Run the Basic Agent</h3>

Run a first prompt through the teaching agent and stream the response while a manual notebook span is active.

In [8]:
first_agent_prompt = (
    'Explain Microsoft Agent Framework to a cloud architect in three short sections: agent creation, tools and MCP, and workflows. '
    'Also include one line about why Aspire Dashboard is useful for local observability.'
)

first_agent_chunks = []
with tracer.start_as_current_span('agent_framework.first_agent_run') as span:
    span.set_attribute('demo.agent.name', teaching_agent.name)
    span.set_attribute('demo.prompt', first_agent_prompt)

    print('Agent (streaming): ', end='')
    async for chunk in teaching_agent.run(first_agent_prompt, stream=True):
        text = getattr(chunk, 'text', None)
        if text:
            print(text, end='', flush=True)
            first_agent_chunks.append(text)

print()
first_agent_response = ''.join(first_agent_chunks).strip()
globals()['first_agent_response'] = first_agent_response
print('\nResponse captured in first_agent_response')

[2026-04-07 04:00:41 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\agent_framework\observability.py:1943 - INFO] {'role': 'system', 'parts': [{'type': 'text', 'content': 'You are a precise but approachable Microsoft Agent Framework teaching assistant. Use tools when they help. Keep explanations concrete, practical, and focused on agents, tools, MCP, workflows, and observability.'}]}
[2026-04-07 04:00:41 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\agent_framework\observability.py:1943 - INFO] {'role': 'user', 'parts': [{'type': 'text', 'content': 'Explain Microsoft Agent Framework to a cloud architect in three short sections: agent creation, tools and MCP, and workflows. Also include one line about why Aspire Dashboard is useful for local observability.'}]}
[2026-04-07 04:00:41 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\agent_framework\observability.py:1943 - INFO] {'role': 'system', 'parts': [{'

Agent (streaming): {
    "body": {
        "role": "system",
        "parts": [
            {
                "type": "text",
                "content": "You are a precise but approachable Microsoft Agent Framework teaching assistant. Use tools when they help. Keep explanations concrete, practical, and focused on agents, tools, MCP, workflows, and observability."
            }
        ]
    },
    "severity_number": 9,
    "severity_text": "INFO",
    "attributes": {
        "event.name": "gen_ai.system.message",
        "gen_ai.provider.name": "microsoft.agent_framework",
        "chat_message_index": 0,
        "code.file.path": "c:\\Users\\lireland\\msft_ai_poc\\foundry_agent_demo\\.venv\\Lib\\site-packages\\agent_framework\\observability.py",
        "code.function.name": "_capture_messages",
        "code.line.number": 1943
    },
    "dropped_attributes": 0,
    "timestamp": "2026-04-07T08:00:41.399094Z",
    "observed_timestamp": "2026-04-07T08:00:41.399208Z",
    "trace_id": "0

[2026-04-07 04:00:44 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\azure\identity\_credentials\azure_cli.py:280 - DEBUG] Executing subprocess with the following arguments ['C:\\Program Files\\Microsoft SDKs\\Azure\\CLI2\\wbin\\az.cmd', 'account', 'get-access-token', '--output', 'json', '--resource', 'https://cognitiveservices.azure.com']
[2026-04-07 04:00:48 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\azure\identity\_internal\decorators.py:24 - INFO] AzureCliCredential.get_token_info succeeded
[2026-04-07 04:00:48 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\azure\identity\_internal\decorators.py:46 - DEBUG] [Authenticated account] Client ID: 04b07795-8ddb-461a-bbee-02f9e1bf7b46. Tenant ID: b22dee98-83da-4207-b9ab-5ba931866f44. User Principal Name: lireland@DibSecurity.onmicrosoft.com. Object ID (user): 1821f16b-41ee-4008-914d-12334122753d
[2026-04-07 04:00:48 - c:\Users\lireland\msft_ai_poc\found

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.namespace": "zolab-agent-framework",
                    "service.instance.id": "fa038a38-9a94-4e28-9867-add1561f834f",
                    "deployment.environment": "demo",
                    "demo.type": "agent-framework-sdk",
                    "service.name": "zolab-agent-framework-sdk-demo",
                    "service.version": "2026.04.07"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
    

[2026-04-07 04:00:50 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\httpcore\_trace.py:87 - DEBUG] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Transfer-Encoding', b'chunked'), (b'Content-Type', b'text/event-stream; charset=utf-8'), (b'skip-error-remapping', b'true'), (b'x-ms-is-spilled-over', b'false'), (b'x-ratelimit-remaining-requests', b'249'), (b'x-ratelimit-reset-requests', b'240'), (b'x-ratelimit-limit-requests', b'250'), (b'x-ratelimit-renewalperiod-requests', b'60'), (b'x-ratelimit-abusepenalty-active', b'False'), (b'x-ratelimit-remaining-tokens', b'249896'), (b'x-ratelimit-reset-tokens', b'24'), (b'x-ratelimit-limit-tokens', b'250000'), (b'x-ratelimit-renewalperiod-tokens', b'60'), (b'x-request-id', b'343cbfbe-1920-45a9-af2b-2cf30a00e395'), (b'apim-request-id', b'03944130-323c-4db5-87f4-1e0bbab1bfd9'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'x-ms-client-request-id', b'Not-S

### 1) Agent creation
Microsoft Agent Framework is centered on building **agents** as a clean abstraction over LLM-driven behavior: you define the agent’s role, instructions, and runtime context, then let it handle task execution and tool use. For a cloud architect, the practical value is that it gives you a repeatable unit for encapsulating domain logic, policy, and interaction patterns.

### 2) Tools and MCP
Agents can call **tools** to fetch data, invoke services, or run actions, and the framework also supports **MCP** so tools can be discovered and integrated in a standard way. That makes it easier to plug agents into existing enterprise systems without hard-coding every dependency into the agent itself.

### 3) Workflows
The framework supports **workflows** and orchestration patterns for chaining agents and steps into multi-stage processes. This is useful when you need human approval, branching logic, or multiple specialized agents cooperating on a task rather than one monolithic 

[2026-04-07 04:00:53 - c:\Users\lireland\msft_ai_poc\foundry_agent_demo\.venv\Lib\site-packages\agent_framework_openai\_chat_client.py:2436 - DEBUG] Unparsed event of type: response.output_text.done: ResponseTextDoneEvent(content_index=0, item_id='msg_0f838116993137450069d4b9b33e9c8197b36f812a3d140567', logprobs=[], output_index=0, sequence_number=239, text='### 1) Agent creation\nMicrosoft Agent Framework is centered on building **agents** as a clean abstraction over LLM-driven behavior: you define the agent’s role, instructions, and runtime context, then let it handle task execution and tool use. For a cloud architect, the practical value is that it gives you a repeatable unit for encapsulating domain logic, policy, and interaction patterns.\n\n### 2) Tools and MCP\nAgents can call **tools** to fetch data, invoke services, or run actions, and the framework also supports **MCP** so tools can be discovered and integrated in a standard way. That makes it easier to plug agents into exist



Response captured in first_agent_response


{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.namespace": "zolab-agent-framework",
                    "service.instance.id": "fa038a38-9a94-4e28-9867-add1561f834f",
                    "deployment.environment": "demo",
                    "demo.type": "agent-framework-sdk",
                    "service.name": "zolab-agent-framework-sdk-demo",
                    "service.version": "2026.04.07"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
    

<h2 style="color: #0078D4;">5. MCP Demo: Expose an Agent Framework Agent as an MCP Server</h2>

This section follows the official <code>microsoft/agent-framework</code> MCP sample pattern: create an Agent Framework agent, call <code>agent.as_mcp_server()</code>, and host it over stdio.

<table style="border: none; border-collapse: collapse; margin-top: 4px;">
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🧰</td>
    <td style="border: none; padding: 4px 0; color: #555;">The notebook writes a small helper script to disk so the MCP server can run in its own process.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">🔌</td>
    <td style="border: none; padding: 4px 0; color: #555;">That helper uses the same Azure OpenAI environment variables and Azure CLI auth model as the rest of the notebook.</td>
  </tr>
  <tr>
    <td style="border: none; padding: 4px 8px 4px 0; vertical-align: top;">📡</td>
    <td style="border: none; padding: 4px 0; color: #555;">When an active trace exists, Agent Framework observability notes that MCP trace context can be propagated automatically to MCP servers.</td>
  </tr>
</table>

In [ ]:
from pathlib import Path
import textwrap

menu_specials_text = "Special Soup: Clam Chowder\nSpecial Salad: Cobb Salad\nSpecial Drink: Chai Tea"

mcp_server_script = Path("agent_framework_menu_mcp_server.py")
script_body = textwrap.dedent(
    f"""
    from typing import Annotated

    import anyio
    from agent_framework import Agent, tool
    from agent_framework.openai import OpenAIChatClient
    from azure.identity.aio import AzureCliCredential

    @tool(approval_mode="never_require")
    def get_specials() -> Annotated[str, "Returns the specials from the menu."]:
        return {menu_specials_text!r}

    @tool(approval_mode="never_require")
    def get_item_price(
        menu_item: Annotated[str, "The name of the menu item."]
    ) -> Annotated[str, "Returns the price of the menu item."]:
        return "$9.99"

    async def run() -> None:
        credential = AzureCliCredential()
        agent = Agent(
            client=OpenAIChatClient(
                model={model_name!r},
                azure_endpoint={azure_openai_endpoint!r},
                credential=credential,
            ),
            name="RestaurantAgent",
            description="Answer questions about the menu.",
            instructions="You are a menu assistant. Answer briefly and use tools when needed.",
            tools=[get_specials, get_item_price],
        )

        server = agent.as_mcp_server()

        from mcp.server.stdio import stdio_server

        async with stdio_server() as (read_stream, write_stream):
            await server.run(read_stream, write_stream, server.create_initialization_options())

    if __name__ == "__main__":
        anyio.run(run)
    """
).lstrip()

mcp_server_script.write_text(script_body, encoding="utf-8")
print(f"Wrote MCP server helper script: {mcp_server_script.resolve()}")

Wrote MCP server helper script: C:\Users\lireland\msft_ai_poc\foundry_agent_demo\agent_framework_menu_mcp_server.py


{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.namespace": "zolab-agent-framework",
                    "service.instance.id": "fa038a38-9a94-4e28-9867-add1561f834f",
                    "deployment.environment": "demo",
                    "demo.type": "agent-framework-sdk",
                    "service.name": "zolab-agent-framework-sdk-demo",
                    "service.version": "2026.04.07"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
    

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

mcp_server_process = globals().get('mcp_server_process')
if mcp_server_process is not None and mcp_server_process.poll() is None:
    print(f'MCP server already running with PID {mcp_server_process.pid}')
else:
    env = os.environ.copy()
    mcp_server_process = subprocess.Popen(
        [sys.executable, str(mcp_server_script)],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        cwd=str(Path.cwd()),
        env=env,
    )
    time.sleep(2)
    if mcp_server_process.poll() is not None:
        stderr_output = (mcp_server_process.stderr.read() or '').strip()
        raise RuntimeError(
            'MCP server failed to stay up. Rerun Cell 19 to regenerate the helper script, then rerun this cell. '
            f'stderr: {stderr_output}'
        )
    globals()['mcp_server_process'] = mcp_server_process
    print(f'Started MCP server with PID {mcp_server_process.pid}')

print('VS Code / MCP host config example:')
print('{')
print('  "servers": {')
print('    "agent-framework-menu": {')
print(f'      "command": "{sys.executable}",')
print(f'      "args": ["{mcp_server_script.resolve()}"]')
print('    }')
print('  }')
print('}')

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.namespace": "zolab-agent-framework",
                    "service.instance.id": "fa038a38-9a94-4e28-9867-add1561f834f",
                    "deployment.environment": "demo",
                    "demo.type": "agent-framework-sdk",
                    "service.name": "zolab-agent-framework-sdk-demo",
                    "service.version": "2026.04.07"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
    

RuntimeError: MCP server failed to stay up. stderr: File [35m"c:\Users\lireland\msft_ai_poc\foundry_agent_demo\agent_framework_menu_mcp_server.py"[0m, line [35m3[0m
    import anyio
[1;35mIndentationError[0m: [35munexpected indent[0m

{
    "resource_metrics": [
        {
            "resource": {
                "attributes": {
                    "telemetry.sdk.language": "python",
                    "telemetry.sdk.name": "opentelemetry",
                    "telemetry.sdk.version": "1.40.0",
                    "service.namespace": "zolab-agent-framework",
                    "service.instance.id": "fa038a38-9a94-4e28-9867-add1561f834f",
                    "deployment.environment": "demo",
                    "demo.type": "agent-framework-sdk",
                    "service.name": "zolab-agent-framework-sdk-demo",
                    "service.version": "2026.04.07"
                },
                "schema_url": ""
            },
            "scope_metrics": [
                {
                    "scope": {
                        "name": "opentelemetry-sdk",
                        "version": null,
                        "schema_url": "",
                        "attributes": null
                    },
    

<h2 style="color: #0078D4;">6. Create a Basic Multi-Agent Workflow</h2>

For the workflow portion, this notebook uses the official <code>GroupChatBuilder</code> pattern from the Agent Framework orchestration docs.

<p style="margin-top: 12px; color: #444;">
  <strong>Architect agent</strong> drafts a plan.<br>
  <strong>Reviewer agent</strong> pressures the plan for security and clarity.<br>
  <strong>Coach agent</strong> rewrites the outcome for a beginner-friendly audience.
</p>

In [ ]:
from agent_framework import Agent, AgentResponseUpdate, Message
from agent_framework.orchestrations import GroupChatBuilder, GroupChatState

architect_agent = Agent(
    client=chat_client,
    name='ArchitectAgent',
    instructions='You design practical Agent Framework demos with tools, MCP, and workflow steps.',
)

reviewer_agent = Agent(
    client=chat_client,
    name='ReviewerAgent',
    instructions='You review proposed demos for technical risk, missing steps, and observability gaps.',
)

coach_agent = Agent(
    client=chat_client,
    name='CoachAgent',
    instructions='You restate technical output so a cloud architect new to Agent Framework can follow it quickly.',
)

def round_robin_selector(state: GroupChatState) -> str:
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]

workflow = GroupChatBuilder(
    participants=[architect_agent, reviewer_agent, coach_agent],
    termination_condition=lambda conv: len(conv) >= 6,
    intermediate_outputs=True,
    selection_func=round_robin_selector,
).build()

globals()['workflow'] = workflow
print('Workflow created with participants: ArchitectAgent, ReviewerAgent, CoachAgent')

In [ ]:
from typing import cast

workflow_task = (
    'Create a short beginner-friendly plan for a Windows-based Agent Framework demo that includes one tool-backed agent, '
    'one MCP example, and one multi-agent workflow, all with strong local observability through Aspire Dashboard.'
)

workflow_transcript = []
with tracer.start_as_current_span('agent_framework.group_chat_workflow') as span:
    span.set_attribute('demo.workflow.type', 'group_chat')
    span.set_attribute('demo.workflow.task', workflow_task)

    async for event in workflow.run(workflow_task, stream=True):
        if event.type != 'output':
            continue

        data = event.data
        if isinstance(data, AgentResponseUpdate):
            if data.text:
                author = data.author_name or 'assistant'
                print(f'{author}: {data.text}', end='', flush=True)
        elif isinstance(data, list):
            print('\n\nFinal Transcript:')
            for msg in cast(list[Message], data):
                if msg.text:
                    line = f'{msg.author_name or msg.role}: {msg.text}'
                    workflow_transcript.append(line)
                    print(line)

globals()['workflow_transcript'] = '\n\n'.join(workflow_transcript)

<h2 style="color: #0078D4;">7. Cleanup</h2>

Use the next cells to stop the MCP server subprocess and optionally close the async Azure credential when you are done with the notebook session.

In [ ]:
mcp_server_process = globals().get('mcp_server_process')
if mcp_server_process is None or mcp_server_process.poll() is not None:
    print('No running MCP server process was found.')
else:
    mcp_server_process.terminate()
    try:
        mcp_server_process.wait(timeout=5)
    except Exception:
        mcp_server_process.kill()
    print('Stopped MCP server process.')

In [ ]:
async_credential = globals().get('async_credential')
if async_credential is None:
    print('Async Azure credential was already cleared.')
else:
    await async_credential.close()
    globals()['async_credential'] = None
    print('Closed AzureCliCredential.')